# 02 — Preprocess and describe the cohort

Filter genera once using the full retained cohort, preserve all metadata, and create exploratory PCA/t-SNE figures.

PCA and t-SNE are descriptive only. TEMPTED and MEFISTO perform their own method-specific transformations later.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from skbio.stats.composition import closure, clr, multi_replace
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# preprocessing settings
PREVALENCE = 0.10
MIN_RA = 0.01
SEED = 2026

root = Path(".") if Path("data").exists() else Path("..")
source = sorted((root / "data" / "processed_16s").iterdir())[-1]
output = root / "data" / "preprocessing" / datetime.now().strftime("%Y%m%d_%H%M%S")
figures = output / "figures"
figures.mkdir(parents=True)

# load outputs from notebook 01
counts = pd.read_csv(source / "counts.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(source / "metadata.csv", dtype={"sample_id": str, "subject_id": str})
taxonomy = pd.read_csv(source / "taxonomy.csv")

## compositional preprocessing

`closure()`       converts each sample to proportions that sum to 1. 
`multi_replace()` performs scikit-bio's standard replacement for zeros
`clr()`           performs the centered log-ratio transform.

These transformations are used only PCA and t-SNE in this notebook.

**scikit-bio composition documentation:**  
https://scikit.bio/docs/latest/generated/skbio.stats.composition.html

**CLR:**  
https://scikit.bio/docs/latest/generated/skbio.stats.composition.clr.html

In [ ]:
# filter genera by prevalence and relative abundance
relative = pd.DataFrame(
    closure(counts),
    index=counts.index,
    columns=counts.columns,
)

keep = (
    (counts.gt(0).mean() >= PREVALENCE)
    & (relative.ge(MIN_RA).mean() >= PREVALENCE)
)
counts = counts.loc[:, keep]
relative = relative.loc[:, keep]

# make clr data use in this notebook's plots
clr_data = clr(multi_replace(relative))

# pca and t-sne
pca = PCA(2).fit_transform(clr_data)
tsne = TSNE(
    2,
    random_state=SEED,
    init="pca",
    learning_rate="auto",
).fit_transform(clr_data)

coords = metadata[["sample_id", "subject_id", "age", "country"]].copy()
coords[["pca_1", "pca_2"]] = pca
coords[["tsne_1", "tsne_2"]] = tsne

In [ ]:
# prepare outputs
counts = counts.reset_index()  # row names to sample_id column

# save outputs
counts.to_csv(output / "counts_filtered.csv", index=False)
metadata.to_csv(output / "metadata.csv", index=False)
coords.to_csv(output / "coordinates.csv", index=False)

print("Retained", counts.shape[1] - 1, "genera")
print("Saved:", output)

In [ ]:
# save a figure

def save(name):
    plt.tight_layout()
    plt.savefig(figures / name, dpi=200)
    plt.show()
    plt.close()

countries = ["FIN", "EST", "RUS"]

colors = {
    "FIN": "green",
    "EST": "blue",
    "RUS": "red",
}

# plot ordinations by country

for x, y, name in [
    ("tsne_1", "tsne_2", "t-SNE"),
    ("pca_1", "pca_2", "PCA"),
]:
    for country in countries:
        rows = coords["country"] == country

        plt.scatter(
            coords.loc[rows, x],
            coords.loc[rows, y],
            s=16,
            alpha=.65,
            label=country,
            color=colors[country],
        )

    plt.xlabel(x)
    plt.ylabel(y)
    plt.legend(frameon=False)
    plt.title(f"{name} by country")
    save(f"{name.lower()}_country.png")

# plot ordinations by age

for x, y, name in [
    ("tsne_1", "tsne_2", "t-SNE"),
    ("pca_1", "pca_2", "PCA"),
]:
    plt.scatter(
        coords[x],
        coords[y],
        c=coords["age"],
        s=16,
        alpha=.7,
        cmap="viridis",
    )

    plt.xlabel(x)
    plt.ylabel(y)
    plt.colorbar(label="Age")
    plt.title(f"{name} by age")
    save(f"{name.lower()}_age.png")

# show cohort size by country

metadata["country"].value_counts().reindex(countries).plot.bar()
plt.ylabel("Samples")
plt.title("Samples by country")
save("samples_by_country.png")

metadata.drop_duplicates("subject_id")["country"].value_counts().reindex(countries).plot.bar()
plt.ylabel("Subjects")
plt.title("Subjects by country")
save("subjects_by_country.png")